# **Análisis crítico**

In [1]:
import pickle
import pandas as pd
import os

## **Tabla comparativa de métricas**

In [2]:
BASE_DIR = "../fasttext_run" 
pickles = {
    "DistilBERT": "distilbert_summary.pkl",
    "Word2Vec_BiLSTM": "word2vec_info.pkl",
    "CNN": "cnn_info.pkl",
    "XGBoost": "xgb_info.pkl",
    "FastText": os.path.join(BASE_DIR, "fasttext_summary.pkl")
}

row_names = [
    "DistilBert Fine Tune",
    "Word2Vec + BiLSTM",
    "CNN-1D",
    "TF-IDF + XGBoost",
    "FastText"
]


def normalize_metrics(metrics_dict):
    return {
        "Accuracy": metrics_dict.get("Accuracy"),
        "Precision (Macro)": metrics_dict.get("Precision (Macro)") or metrics_dict.get("Precision_Macro"),
        "Recall (Macro)": metrics_dict.get("Recall (Macro)") or metrics_dict.get("Recall_Macro"),
        "F1-Score (Macro)": metrics_dict.get("F1-Score (Macro)") or metrics_dict.get("F1_Macro"),
        "ROC AUC (Macro)": metrics_dict.get("ROC AUC (Macro)") or metrics_dict.get("ROC_AUC_Macro")
    }

rows = []
for path in pickles.values():
    with open(path, "rb") as f:
        info = pickle.load(f)
    metrics = normalize_metrics(info["metrics_test"])
    rows.append(metrics)

resultados = pd.DataFrame(rows, index=row_names)

resultados


2025-11-08 17:33:59.401786: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-11-08 17:33:59.467338: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-11-08 17:34:00.847528: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1762641243.142228    2763 gpu_device.cc:2020] Created device /job:localhost/rep

,Accuracy,Precision (Macro),Recall (Macro),F1-Score (Macro),ROC AUC (Macro)
DistilBert Fine Tune,0.823056,0.772386,0.772137,0.766004,0.957838
Word2Vec + BiLSTM,0.756032,0.702512,0.705388,0.696467,0.945301
CNN-1D,0.788204,0.764300,0.740643,0.739106,0.960109
TF-IDF + XGBoost,0.825737,0.791440,0.780680,0.777748,0.982200
FastText,0.672922,0.646081,0.632390,0.631480,0.953611


Para la realización de este ejercicio, se escogieron como métricas de interés **F1-Score Macro** y **ROC AUC Macro**, con el fin de evaluar el desempeño general de los modelos, considerando tanto la precisión en cada clase como la capacidad global de discriminación entre clases.

A partir de la tabla comparativa de métricas, se observa que el mejor modelo en este caso fue **`TF-IDF + XGBoost`**, alcanzando los valores más altos en ambas métricas. Esto indica que, en términos generales, este modelo logra un buen equilibrio entre precisión y recall en todas las clases, así como una sólida capacidad de clasificación multiclase según corresponda.

También se observa que, en general, todos los modelos lograron distinguir bien entre las clases, ya que los valores de **ROC AUC** superan los **0.9** en todos los casos, siendo el más alto de **0.98**, lo que refleja un desempeño notable en términos de discriminación.

### **Posibles razones**

El buen rendimiento del modelo TF-IDF combinado con XGBoost puede atribuirse a diversos factores.

Una de las razones es que este modelo fue el único que no utilizó embeddings ni modelos preentrenados, lo cual pudo haber sido beneficioso. Al trabajar directamente con las palabras del corpus, logró captar información específica y términos poco frecuentes que quizás no están bien representados en los vocabularios de modelos preentrenados, lo que favoreció su desempeño en este conjunto de datos en particular.

Además, XGBoost es especialmente eficaz al manejar representaciones textuales extensas y dispersas, ya que puede identificar automáticamente las palabras más relevantes para distinguir entre clases. También tiende a ser menos susceptible al sobreajuste en comparación con modelos más complejos como los transformers o las redes neuronales, lo que contribuye a una mayor estabilidad en sus predicciones.

Por otro lado, cada palabra se representa de forma explícita, mientras que los modelos preentrenados pueden llegar a cortar las palabras poco comunes en partes, lo que podría llevar a una pérdida de significado.


### **Limitaciones**

En este proyecto, identifiqué dos limitaciones frecuentes en los cinco modelos evaluados:

* **Clases minoritarias**: Durante el análisis exploratorio de datos (EDA) se identificó un desbalance en la base de datos. Para reducirlo, se aplicó oversampling únicamente sobre las clases minoritarias como una forma de data augmentation para mejorar los modelos. No obstante, la categoría BPO, que contaba con la menor cantidad de curriculums desde un inicio, continuó presentando dificultades en la clasificación y, en general, obtuvo el peor desempeño entre las distintas clases. Este tipo de situaciones representa una limitación importante tanto para los modelos como para la obtención de mejores resultados.

* **Capacidad computacional**: Otra limitación significativa fue la capacidad de mi computador, incluyendo GPU, VRAM y procesador. Debido al tamaño del dataset y a la necesidad de realizar hiperparametrización, en varios momentos tuve que reducir procesos o simplificar configuraciones, lo que limitó las oportunidades de entrenar algunos modelos de manera óptima y, posiblemente, de obtener resultados más altos.

### **Recomendaciones de mejora**

* **Balanceo entre clases**: Explorar técnicas adicionales como combinaciones de oversampling y undersampling, o aplicar data augmentation más intensivo en las clases minoritarias. Esto puede contribuir a una mejor representación de dichas clases y mejorar el rendimiento de los modelos en la clasificación de casos menos frecuentes.


* **Optimización de recursos computacionales**: Para facilitar el entrenamiento de modelos más complejos y permitir una búsqueda de hiperparámetros más exhaustiva, se pueden aplicar alternativas tales como reducir el tamaño de los lotes, optimizar el preprocesamiento del texto, o considerar el uso de computadores con capacidades más potentes.


* **Mejora en la comprensión contextual**: Se podría probar con modelos que entienden mejor el significado de las palabras según el entorno en el que aparecen, como los que usan inteligencia artificial avanzada. Además, combinar varios modelos diferentes puede ayudar a que el sistema sea más preciso y confiable.